In [2]:
# pip install pydeseq2 gseapy pandas numpy scipy statsmodels torch

from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch

from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats  # contrast accepts [var, tested_level, reference_level] :contentReference[oaicite:2]{index=2}

import gseapy as gp  # gene_sets can be a dict of {set_name: [genes...]} :contentReference[oaicite:3]{index=3}


@dataclass
class DE_ORA_GSEA_Result:
    de_results: pd.DataFrame
    de_gene_list: List[str]
    ora_results: pd.DataFrame
    gsea_results: pd.DataFrame


def build_gene_sets_from_mask(
    mask: torch.Tensor,
    gene_names: List[str],
    pathway_names: List[str],
    min_genes_per_set: int = 5,
) -> Dict[str, List[str]]:
    """
    Convert (n_genes, n_pathways) binary mask into a gene_sets dict.
    """
    if mask.ndim != 2:
        raise ValueError(f"mask must be 2D (n_genes, n_pathways), got shape {tuple(mask.shape)}")

    n_genes, n_pathways = mask.shape
    if len(gene_names) != n_genes:
        raise ValueError("len(gene_names) must equal mask.shape[0]")
    if len(pathway_names) != n_pathways:
        raise ValueError("len(pathway_names) must equal mask.shape[1]")

    m = mask.detach().cpu().to(torch.bool)
    gene_sets: Dict[str, List[str]] = {}
    for p in range(n_pathways):
        genes_p = [gene_names[g] for g in torch.where(m[:, p])[0].tolist()]
        if len(genes_p) >= min_genes_per_set:
            gene_sets[pathway_names[p]] = genes_p
    return gene_sets


def run_deseq2_two_class(
    x_counts: torch.Tensor,
    y: torch.Tensor,
    gene_names: List[str],
    label_test: int,
    label_ref: int,
    n_cpus: int = 8,
    alpha: float = 0.05,
) -> pd.DataFrame:
    """
    Step 1) Differential expression via PyDESeq2 for a two-class comparison.
    Returns DE results DataFrame with gene names as index.
    """
    if x_counts.ndim != 2:
        raise ValueError("x_counts must be (n_samples, n_genes)")
    if y.ndim != 1:
        raise ValueError("y must be (n_samples,)")

    x = x_counts.detach().cpu().to(torch.int64).numpy()
    y_np = y.detach().cpu().numpy()

    # Filter to the two classes we’re comparing
    keep = np.isin(y_np, [label_test, label_ref])
    x = x[keep]
    y_np = y_np[keep]

    # Build DESeq2 inputs
    counts_df = pd.DataFrame(x, columns=gene_names)
    # PyDESeq2 expects sample metadata (clinical) with a factor column used in the design :contentReference[oaicite:4]{index=4}
    clinical_df = pd.DataFrame(
        {"condition": pd.Categorical(y_np.astype(str))},
        index=counts_df.index,
    )

    # Fit DESeq2-like model
    dds = DeseqDataSet(
        counts=counts_df,
        clinical=clinical_df,
        design="~ condition",  # simple one-factor design :contentReference[oaicite:5]{index=5}
        n_cpus=n_cpus,
    )
    dds.deseq2()

    # Contrast format: [variable_of_interest, tested_level, reference_level] :contentReference[oaicite:6]{index=6}
    contrast = ["condition", str(label_test), str(label_ref)]
    stat_res = DeseqStats(dds, contrast=contrast, alpha=alpha, n_cpus=n_cpus)  # :contentReference[oaicite:7]{index=7}
    stat_res.summary()

    res = stat_res.results_df.copy()
    # Ensure gene names are index
    res.index.name = "gene"
    return res


def select_de_genes(
    de_df: pd.DataFrame,
    padj_col: str = "padj",
    lfc_col: str = "log2FoldChange",
    alpha: float = 0.05,
    min_abs_log2fc: float = 0.0,
) -> List[str]:
    """
    Step 2) Select DE genes by adjusted p-value and optional log2FC threshold.
    """
    if padj_col not in de_df.columns:
        raise KeyError(f"Missing {padj_col} in DE results columns: {list(de_df.columns)}")
    if lfc_col not in de_df.columns:
        raise KeyError(f"Missing {lfc_col} in DE results columns: {list(de_df.columns)}")

    m = de_df[padj_col].notna()
    m &= (de_df[padj_col] <= alpha)
    if min_abs_log2fc > 0:
        m &= (de_df[lfc_col].abs() >= min_abs_log2fc)

    return de_df.index[m].astype(str).tolist()


def ora_from_mask(
    de_genes: List[str],
    gene_names: List[str],
    mask: torch.Tensor,
    pathway_names: List[str],
    alpha: float = 0.05,
    alternative: str = "greater",
) -> pd.DataFrame:
    """
    Step 3) ORA using Fisher exact test per pathway directly from membership mask.
    Universe = all genes in gene_names.
    """
    gene_to_idx = {g: i for i, g in enumerate(gene_names)}
    de_idx = np.array([gene_to_idx[g] for g in de_genes if g in gene_to_idx], dtype=int)

    m = mask.detach().cpu().to(torch.bool).numpy()  # (n_genes, n_pathways)
    n_genes, n_pathways = m.shape

    de_set = np.zeros(n_genes, dtype=bool)
    de_set[de_idx] = True
    n_de = int(de_set.sum())
    n_not_de = n_genes - n_de

    rows = []
    for p in range(n_pathways):
        in_path = m[:, p]
        a = int((de_set & in_path).sum())             # DE and in pathway
        b = int((~de_set & in_path).sum())            # not DE and in pathway
        c = int((de_set & ~in_path).sum())            # DE and not in pathway
        d = int((~de_set & ~in_path).sum())           # not DE and not in pathway

        # Skip empty pathways
        if (a + b) == 0:
            continue

        # Fisher exact ORA
        oddsratio, pval = fisher_exact([[a, b], [c, d]], alternative=alternative)

        rows.append(
            {
                "pathway": pathway_names[p],
                "pathway_size": int(in_path.sum()),
                "de_in_pathway": a,
                "oddsratio": oddsratio,
                "pval": pval,
            }
        )

    ora = pd.DataFrame(rows)
    if len(ora) == 0:
        ora["padj"] = []
        return ora

    # BH-FDR
    ora["padj"] = multipletests(ora["pval"].values, method="fdr_bh")[1]
    ora = ora.sort_values(["padj", "pval"], ascending=[True, True]).reset_index(drop=True)
    ora["significant"] = ora["padj"] <= alpha
    return ora


def gsea_prerank_from_de(
    de_df: pd.DataFrame,
    gene_sets: Dict[str, List[str]],
    score_col_candidates: Tuple[str, ...] = ("stat", "statistic"),
    fallback_use: str = "signed_log10p",
    min_size: int = 5,
    max_size: int = 5000,
    permutation_num: int = 1000,
    seed: int = 0,
) -> pd.DataFrame:
    """
    Step 4) Pre-ranked GSEA using GSEApy.
    We prefer a Wald-like statistic column if present; otherwise build a fallback rank score.
    GSEApy supports gene_sets as a dict :contentReference[oaicite:8]{index=8}
    """
    df = de_df.copy()

    # Choose ranking score
    score_col = None
    for c in score_col_candidates:
        if c in df.columns and np.isfinite(df[c]).any():
            score_col = c
            break

    if score_col is None:
        if fallback_use == "signed_log10p":
            # rank = sign(log2FC) * -log10(pvalue)
            if "log2FoldChange" not in df.columns or "pvalue" not in df.columns:
                raise KeyError("Need log2FoldChange and pvalue columns for fallback ranking")
            p = df["pvalue"].astype(float).clip(lower=1e-300)
            s = np.sign(df["log2FoldChange"].astype(float).fillna(0.0))
            df["_rank_score"] = s * (-np.log10(p))
            score_col = "_rank_score"
        else:
            raise ValueError(f"Unknown fallback_use={fallback_use}")

    # Build rnk as a two-column DataFrame: [gene, score]
    rnk = (
        df[[score_col]]
        .rename(columns={score_col: "score"})
        .assign(gene=df.index.astype(str))
        .loc[:, ["gene", "score"]]
        .dropna()
    )

    # Remove duplicates by taking max absolute score
    rnk = rnk.sort_values("score", ascending=False)
    rnk = rnk.drop_duplicates("gene", keep="first")

    # Run prerank
    pre = gp.prerank(
        rnk=rnk,
        gene_sets=gene_sets,        # dict is supported :contentReference[oaicite:9]{index=9}
        min_size=min_size,
        max_size=max_size,
        permutation_num=permutation_num,
        seed=seed,
        outdir=None,                # no files; keep in-memory
        verbose=False,
    )
    # GSEApy returns results in pre.res2d
    return pre.res2d.copy()


def de_ora_gsea_pipeline(
    x: torch.Tensor,
    y: torch.Tensor,
    mask: torch.Tensor,
    gene_names: List[str],
    pathway_names: List[str],
    label_test: int,
    label_ref: int,
    alpha: float = 0.05,
    min_abs_log2fc: float = 0.0,
    n_cpus: int = 8,
    gsea_permutation_num: int = 1000,
) -> DE_ORA_GSEA_Result:
    """
    Runs steps 1-4:
      1) DE with PyDESeq2
      2) Select DE genes
      3) ORA with Fisher exact from mask
      4) GSEA prerank with GSEApy
    """
    # 1) DE
    de_df = run_deseq2_two_class(
        x_counts=x,
        y=y,
        gene_names=gene_names,
        label_test=label_test,
        label_ref=label_ref,
        n_cpus=n_cpus,
        alpha=alpha,
    )

    # 2) Select DE genes
    de_genes = select_de_genes(
        de_df,
        alpha=alpha,
        min_abs_log2fc=min_abs_log2fc,
    )

    # Gene sets from mask
    gene_sets = build_gene_sets_from_mask(
        mask=mask,
        gene_names=gene_names,
        pathway_names=pathway_names,
        min_genes_per_set=5,
    )

    # 3) ORA
    ora_df = ora_from_mask(
        de_genes=de_genes,
        gene_names=gene_names,
        mask=mask,
        pathway_names=pathway_names,
        alpha=alpha,
        alternative="greater",
    )

    # 4) GSEA (prerank)
    gsea_df = gsea_prerank_from_de(
        de_df=de_df,
        gene_sets=gene_sets,
        permutation_num=gsea_permutation_num,
        seed=0,
    )

    return DE_ORA_GSEA_Result(
        de_results=de_df,
        de_gene_list=de_genes,
        ora_results=ora_df,
        gsea_results=gsea_df,
    )


# -------------------------
# Example usage
# -------------------------
# x: torch.Tensor (n_samples, n_genes) raw counts (integers)
# y: torch.Tensor (n_samples,) int labels
# mask: torch.Tensor (n_genes, n_pathways) binary
# gene_names: List[str] length n_genes (e.g., HGNC symbols or Ensembl IDs)
# pathway_names: List[str] length n_pathways

# res = de_ora_gsea_pipeline(
#     x=x,
#     y=y,
#     mask=mask,
#     gene_names=gene_names,
#     pathway_names=pathway_names,
#     label_test=1,     # e.g., tumor
#     label_ref=0,      # e.g., normal
#     alpha=0.05,
#     min_abs_log2fc=0.0,
#     n_cpus=8,
#     gsea_permutation_num=1000,
# )
#
# res.de_results.head()
# res.ora_results.head()
# res.gsea_results.head()
# len(res.de_gene_list)


In [ ]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device(drop=['cuda:4'])

#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
_batch = d.get_toy_databatch(dataset, device)

/home/mv18gs/miniconda3/envs/thesis_pyg/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


('cuda:4', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:6', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 78941)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 74623)
('cuda:1', 'NVIDIA A100-SXM4-80GB', 57347)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 38977)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 38975)

# #### Device() ####
# device = cuda:6

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:6)
# edge_attr                (32464, 16)              Tensor (cuda:6)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:6)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path            

In [ ]:
print('hi')

hi


In [ ]:
dataset

GraphDataset(1172)

In [ ]:
dataset.x.shape

torch.Size([1172, 4373, 1])

In [ ]:
dataset.y.shape

torch.Size([1172])

In [ ]:
dataset.wrapper.pathway_index

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:6')

In [ ]:
dataset.wrapper.pathway_labels

['path:hsa04975',
 'path:hsa04010',
 'path:hsa04630',
 'path:hsa05030',
 'path:hsa05210',
 'path:hsa04137',
 'path:hsa05169',
 'path:hsa04114',
 'path:hsa05205',
 'path:hsa04520',
 'path:hsa05320',
 'path:hsa04666',
 'path:hsa00280',
 'path:hsa04672',
 'path:hsa05330',
 'path:hsa04068',
 'path:hsa05412',
 'path:hsa04668',
 'path:hsa00520',
 'path:hsa04061',
 'path:hsa00750',
 'path:hsa00563',
 'path:hsa05321',
 'path:hsa04613',
 'path:hsa00592',
 'path:hsa04979',
 'path:hsa04340',
 'path:hsa00240',
 'path:hsa04614',
 'path:hsa00030',
 'path:hsa00565',
 'path:hsa05222',
 'path:hsa04934',
 'path:hsa04911',
 'path:hsa04976',
 'path:hsa05417',
 'path:hsa04122',
 'path:hsa00630',
 'path:hsa00440',
 'path:hsa04915',
 'path:hsa04360',
 'path:hsa04371',
 'path:hsa05110',
 'path:hsa03018',
 'path:hsa00100',
 'path:hsa00670',
 'path:hsa00330',
 'path:hsa05031',
 'path:hsa05322',
 'path:hsa04512',
 'path:hsa05034',
 'path:hsa04964',
 'path:hsa04622',
 'path:hsa05150',
 'path:hsa00400',
 'path:hsa

In [ ]:
dataset.wrapper.x_labels

['FGR',
 'GCLC',
 'NFYA',
 'SEMA3F',
 'CFTR',
 'CYP51A1',
 'KRIT1',
 'BAD',
 'LAP3',
 'AOC1',
 'M6PR',
 'CYP26B1',
 'ALS2',
 'CASP10',
 'CFLAR',
 'MTMR7',
 'SARM1',
 'AK2',
 'CD38',
 'KDM1A',
 'CAMKK1',
 'NDUFAB1',
 'PDK4',
 'CDC27',
 'HCCS',
 'DVL2',
 'UPF1',
 'SLC25A5',
 'DHX33',
 'ACSM3',
 'PRKAR2B',
 'CREBBP',
 'KMT2E',
 'ZNF195',
 'ITGAL',
 'PDK2',
 'ITGA3',
 'LAMP2',
 'ITGA2B',
 'MAP3K14',
 'ABCC8',
 'AP2B1',
 'CX3CL1',
 'CACNA1G',
 'TNFRSF12A',
 'MAP3K9',
 'RALA',
 'AGK',
 'ALDH3B1',
 'FARP2',
 'GGCT',
 'TBXA2R',
 'COX10',
 'VPS41',
 'SCIN',
 'PNPLA4',
 'ADIPOR2',
 'PAFAH1B1',
 'DNAH9',
 'MATK',
 'CD79B',
 'RHBDF1',
 'CACNA2D2',
 'TEAD3',
 'SELE',
 'FMO3',
 'MYLIP',
 'NOX1',
 'E2F2',
 'PSMB1',
 'NADK',
 'CYTH3',
 'AASS',
 'MGST1',
 'ST3GAL1',
 'MMP25',
 'MAPK8IP2',
 'MASP2',
 'POMT2',
 'VTA1',
 'MLXIPL',
 'UQCRC1',
 'GIPR',
 'SEMA3G',
 'STAB1',
 'IDS',
 'ZNF200',
 'CD4',
 'BTK',
 'HFE',
 'FYN',
 'FMO1',
 'LYPLA2',
 'MRC2',
 'ABHD5',
 'PIK3C2A',
 'PLAUR',
 'DCN',
 'PPP5C',
 'MAP4